# 01 · Tabular data and an honest baseline

**ML Science Lab · hands-on notebook**

**Learning goals:** identify rows, features and targets; distinguish training, validation and test; compare simple baselines; avoid data leakage.

Dataset: scikit-learn Wine (178 wine samples, 13 numerical chemistry measurements, 3 cultivar classes). Run cells top-to-bottom. Leave the final 20% test set untouched in this notebook.

In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

data = load_wine(as_frame=True)
X, y = data.data, data.target
print(f"Rows: {len(X)} | Features: {X.shape[1]} | Classes: {y.nunique()}")
display(X.head())
print('Class sizes:', y.value_counts().sort_index().to_dict())

### 1. Prepare the split

Keep **20% as an untouched final test set**. Split the remaining 80% into 60% for fitting and 20% for choosing model settings. The split is stratified, preserving the class composition.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=42, stratify=y_dev
)
print(f'Train {len(X_train)} | Validation {len(X_val)} | Reserved test {len(X_test)}')

### 2. Fit a simple baseline

The dummy classifier always predicts the most frequent class. A useful ML model should improve upon it. We report accuracy **and** macro-F1 (class-averaged F1).

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=2)

for name, estimator in [('Most-frequent baseline', dummy), ('Random forest', rf)]:
    estimator.fit(X_train, y_train)
    predicted = estimator.predict(X_val)
    print(f'{name:24s} accuracy={accuracy_score(y_val, predicted):.3f} '
          f'macro-F1={f1_score(y_val, predicted, average="macro"):.3f}')

### 3. Your experiment

Change **only** `max_depth` below (e.g. 2, 5, 12) and run this cell again. Do higher values always help? Why is validation accuracy not the same as test accuracy?

In [ ]:
max_depth = 5  # YOUR TURN: try 2 and 12
model = RandomForestClassifier(n_estimators=100, max_depth=max_depth, random_state=42, n_jobs=2)
model.fit(X_train, y_train)
predicted = model.predict(X_val)
print(f'Validation macro-F1 = {f1_score(y_val, predicted, average="macro"):.3f}')
ConfusionMatrixDisplay.from_predictions(y_val, predicted)
plt.title('Validation set (NOT the test set)')
plt.show()

**Discuss in pairs:** Why is a random split wrong when samples are repeated measurements of the same person, specimen or experiment? What happens when feature preprocessing is fit to the entire dataset? **Do not choose hyperparameters by repeatedly checking `X_test`.**

Next: [02 — AutoML classification](02_automl_classification.ipynb) (or return to the workshop's Colab links).